<a href="https://colab.research.google.com/github/ankit-thawal47/60-days-inference-engineering/blob/main/Day4/Day_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Build a basic MLP(Multi Layer Perceptron) Model


In [1]:
import torch
import matplotlib.pyplot as plt
from torch import nn

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
X,y = make_moons(n_samples = 100, noise = 0.2, random_state = 42)

In [ ]:
X = torch.tensor(X, dtype = torch.float32)
y = torch.tensor(y, dtype = torch.float32)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
class SimpleModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.network = nn.Sequential(
        nn.Linear(2, 4),
        nn.ReLU(),
        nn.Linear(4, 1),
        nn.Sigmoid()
    )

  def forward(self, x):
    return self.network(x)


In [ ]:
model = SimpleModel()
print(model)

SimpleModel(
  (network): Sequential(
    (0): Linear(in_features=2, out_features=4, bias=True)
    (1): ReLU()
    (2): Linear(in_features=4, out_features=1, bias=True)
    (3): Sigmoid()
  )
)


In [ ]:
total = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total}")

Total parameters: 17


In [ ]:
output = model(X)
print(f"Input shape:  {X.shape}")      # (200, 2)
print(f"Output shape: {output.shape}") # (200, 1)
print(f"Sample outputs: {output[:5]}")  # raw probabilities 0-1

Input shape:  torch.Size([100, 2])
Output shape: torch.Size([100, 1])
Sample outputs: tensor([[0.4705],
        [0.4340],
        [0.4340],
        [0.3976],
        [0.4088]], grad_fn=<SliceBackward0>)


In [ ]:
# This proves autograd is ready — no training yet, just checking
loss = output.mean()
loss.backward()  # should work without any requires_grad calls

# Check gradients were computed on weights
for name, param in model.named_parameters():
    print(f"{name:40s} grad shape: {param.grad.shape}")

network.0.weight                         grad shape: torch.Size([4, 2])
network.0.bias                           grad shape: torch.Size([4])
network.2.weight                         grad shape: torch.Size([1, 4])
network.2.bias                           grad shape: torch.Size([1])


TRAINING ON MNIST DATASET

In [ ]:
class MLPNetwork(nn.Module):

  def __init__(self):
    super().__init__()
    self.network = nn.Sequential(
        nn.Linear(784, 128),   # flatten input → hidden layer
        nn.ReLU(),             # non-linearity
        nn.Linear(128, 10),    # hidden → 10 class scores
        nn.Softmax(dim=1)
    )


  def forward(self, x):
    return self.network(x)

In [ ]:
#1. Load the dataset

#2. Split the database into tranining and test

#3. get the trainign dataset from the network and let it compute the weights, I think we have to decide the epoch as well

#4. the testing dataset is then used to find the "correctness", if the deviation is too much then increase the epoch and train it again

In [1]:
import torch
from torch import nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [2]:
transform = transforms.Compose([
    transforms.ToTensor()
])

train_data = datasets.MNIST(root='data', train=True,  download=True, transform=transform)
test_data  = datasets.MNIST(root='data', train=False, download=True, transform=transform)

print(f"Training samples: {len(train_data)}")
print(f"Test samples:     {len(test_data)}")

100%|██████████| 9.91M/9.91M [00:01<00:00, 6.55MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 154kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.47MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 6.17MB/s]

Training samples: 60000
Test samples:     10000


In [4]:
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_data,  batch_size=64, shuffle=False)

# Peek at one batch
X_batch, y_batch = next(iter(train_loader))
print(f"Batch input shape: {X_batch.shape}")   # (64, 1, 28, 28)
print(f"Batch label shape: {y_batch.shape}")   # (64,)

Batch input shape: torch.Size([64, 1, 28, 28])
Batch label shape: torch.Size([64])


In [6]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [5]:
class MLPNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),            # (64,1,28,28) → (64,784)
            nn.Linear(784, 128),     # input → hidden
            nn.ReLU(),               # non-linearity
            nn.Linear(128, 10),      # hidden → 10 classes
        )

    def forward(self, x):
        return self.network(x)

model = MLPNetwork()
print(model)

MLPNetwork(
  (network): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=784, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=10, bias=True)
  )
)


In [7]:
epochs = 5

for epoch in range(epochs):
  model.train()
  total_loss = 0

  for X_batch, Y_batch in train_loader:
    optimizer.zero_grad()
    output = model(X_batch)
    loss = criterion(output, Y_batch)
    loss.backward()
    optimizer.step()
    total_loss += loss.item()

  print(f"Epoch {epoch+1}/{epochs}  Loss: {total_loss/len(train_loader):.4f}")


Epoch 1/5  Loss: 0.3383
Epoch 2/5  Loss: 0.1532
Epoch 3/5  Loss: 0.1073
Epoch 4/5  Loss: 0.0811
Epoch 5/5  Loss: 0.0640
